# Load test – Coffee Shop Order Management (Kaggle CPU)



## 1. Kiểm tra môi trường

In [ ]:
!nproc
!free -h
!head -2 /etc/os-release

## 2. Cấu hình

In [ ]:
import os, secrets, base64, subprocess, time

# Cách A: clone từ GitHub
REPO_URL = "https://github.com/dqh1215/Coffee-shop-order-management.git"
BRANCH   = "dev"
# Cách B: upload source zip thành Kaggle Dataset rồi điền đường dẫn, ví dụ "/kaggle/input/coffeeshop-src"
DATASET_DIR = ""

APP_DIR  = "/kaggle/working/app"
JDK_DIR  = "/opt/jdk"
RESULTS  = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

USER_LEVELS = [50, 100, 200]   # số người dùng ảo mỗi kịch bản
SPAWN_RATE  = 10               # user/giây
DURATION    = "3m"

NCPU = len(os.sched_getaffinity(0))
APP_CORES    = f"0-{NCPU-2}"   # Java + MySQL
LOCUST_CORES = f"{NCPU-1}"     # Locust
print("CPU:", NCPU, "| app cores:", APP_CORES, "| locust core:", LOCUST_CORES)

## 3. Lấy source code

In [ ]:
if DATASET_DIR:
    !rm -rf {APP_DIR} && cp -r {DATASET_DIR} {APP_DIR}
else:
    !rm -rf {APP_DIR} && git clone --depth 1 -b {BRANCH} {REPO_URL} {APP_DIR}
!ls {APP_DIR}

## 4. Cài JDK 25 (Temurin)

In [ ]:
!mkdir -p {JDK_DIR} && curl -sL "https://api.adoptium.net/v3/binary/latest/25/ga/linux/x64/jdk/hotspot/normal/eclipse" | tar -xz -C {JDK_DIR} --strip-components=1
!{JDK_DIR}/bin/java -version

## 5. Cài và khởi động MySQL, nạp schema + seed

In [ ]:
!apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq mysql-server > /dev/null
!mkdir -p /var/run/mysqld && chown mysql:mysql /var/run/mysqld

mysqld = subprocess.Popen(f"taskset -c {APP_CORES} mysqld_safe",
                          shell=True, stdout=open("/tmp/mysqld.log","w"), stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run("mysqladmin ping --silent", shell=True).returncode == 0:
        print("MySQL ready"); break
    time.sleep(1)

!mysql -uroot -e "DROP DATABASE IF EXISTS coffeeshop; CREATE DATABASE coffeeshop; CREATE USER IF NOT EXISTS 'coffee'@'%' IDENTIFIED BY 'coffeepass'; GRANT ALL ON coffeeshop.* TO 'coffee'@'%'; FLUSH PRIVILEGES;"
!mysql -uroot coffeeshop < {APP_DIR}/infrastructure/database/init/01-schema.sql
!mysql -uroot coffeeshop -e "SELECT name, price FROM menu_items;"

## 6. Build jar

In [ ]:
!cd {APP_DIR} && chmod +x mvnw && JAVA_HOME={JDK_DIR} ./mvnw -q -DskipTests package
!ls -lh {APP_DIR}/target/*.jar

## 7. Khởi động app

In [ ]:
import glob, requests
jar = [j for j in glob.glob(f"{APP_DIR}/target/*.jar") if not j.endswith("plain.jar")][0]

env = os.environ.copy()
env.update({
    "DB_HOST": "127.0.0.1",
    "DB_PORT": "3306",
    "MYSQL_DATABASE": "coffeeshop",
    "MYSQL_USERNAME": "coffee",
    "MYSQL_ROOT_PASSWORD": "coffeepass",
    "SPRING_DATASOURCE_URL": "jdbc:mysql://127.0.0.1:3306/coffeeshop?allowPublicKeyRetrieval=true&useSSL=false",
    "JWT_SECRET": base64.b64encode(secrets.token_bytes(64)).decode(),  # sinh ngẫu nhiên, không lưu lại
    "JWT_EXPIRATION_MS": "86400000",
    "TAX_RATE": "0",
    "ADMIN_EMAIL": "admin@coffeeshop.local",
    "ADMIN_PASSWORD": secrets.token_urlsafe(16),   # mật khẩu admin ngẫu nhiên, chỉ dùng cho phiên test
})
cmd = (f"taskset -c {APP_CORES} {JDK_DIR}/bin/java -Xms1g -Xmx2g -jar {jar} "
       "--spring.jpa.show-sql=false --logging.level.root=WARN")
app = subprocess.Popen(cmd, shell=True, env=env, stdout=open("/tmp/app.log","w"), stderr=subprocess.STDOUT)

BASE = "http://localhost:8080"
for _ in range(120):
    try:
        if requests.get(f"{BASE}/api/menu-items", timeout=2).status_code == 200:
            print("App ready"); break
    except Exception:
        pass
    time.sleep(1)
else:
    print(open("/tmp/app.log").read()[-5000:])

## 8. Smoke test

In [ ]:
d = requests.get(f"{BASE}/api/menu-items").json()["data"]
items = d["items"] if isinstance(d, dict) else d
print("menu:", len(items), "items")
u = {"email":"smoke@example.com","password":"Smoke@1234","confirm_password":"Smoke@1234","full_name":"Smoke"}
print("register:", requests.post(f"{BASE}/api/auth/register", json=u).status_code)
r = requests.post(f"{BASE}/api/auth/login", json={"email":u["email"],"password":u["password"]})
print("login:", r.status_code)
h = {"Authorization": f"Bearer {r.json()['data']['access_token']}"}
o = requests.post(f"{BASE}/api/orders", headers=h, json={
    "items": [{"menu_item_id": "913ea87e-d553-408e-b031-478d1cf8f949", "quantity": 2}],
    "voucher_code": "WELCOME10", "note": "smoke"})
print("order:", o.status_code, o.json()["data"]["total_amount"] if o.ok else o.text)

## 8b. Warm-up JVM (kết quả bỏ đi)

Chạy ngắn để JIT/cache nóng lên, tránh làm lệch mức tải đầu tiên và burst test.

In [ ]:
subprocess.run(
    f"RUN_ID=warm taskset -c {LOCUST_CORES} locust -f {APP_DIR}/loadtest/locustfile.py --host {BASE} --headless "
    f"-u 20 -r 10 -t 30s --only-summary",
    shell=True)
time.sleep(5)


## 9. Chạy load test (3 mức tải)

In [ ]:
!pip install -q locust
import psutil, threading, statistics

LOCUSTFILE = f"{APP_DIR}/loadtest/locustfile.py"
cpu_summary = {}

def sample_cpu(stop, out):
    while not stop.is_set():
        out.append(psutil.cpu_percent(interval=1, percpu=True))

for users in USER_LEVELS:
    prefix = f"{RESULTS}/u{users}"
    samples, stop = [], threading.Event()
    t = threading.Thread(target=sample_cpu, args=(stop, samples)); t.start()
    print(f"=== {users} users, {DURATION} ===")
    subprocess.run(
        f"RUN_ID=u{users} taskset -c {LOCUST_CORES} locust -f {LOCUSTFILE} --host {BASE} --headless "
        f"-u {users} -r {SPAWN_RATE} -t {DURATION} --csv {prefix} --html {prefix}.html --only-summary",
        shell=True)
    stop.set(); t.join()
    app_cores = range(NCPU - 1)
    per_sec = [statistics.mean(s[i] for i in app_cores) for s in samples]
    cpu_summary[users] = (statistics.mean(per_sec), max(per_sec))
    time.sleep(15)  # nghỉ giữa các kịch bản

## 10. Tổng hợp kết quả

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

rows, per_endpoint = [], []
for users in USER_LEVELS:
    df = pd.read_csv(f"{RESULTS}/u{users}_stats.csv")
    agg = df[df["Name"] == "Aggregated"].iloc[0]
    rows.append({
        "users": users,
        "requests": int(agg["Request Count"]),
        "failures": int(agg["Failure Count"]),
        "error_%": round(100 * agg["Failure Count"] / max(agg["Request Count"], 1), 2),
        "RPS": round(agg["Requests/s"], 1),
        "avg_ms": round(agg["Average Response Time"], 1),
        "p50_ms": agg["50%"], "p95_ms": agg["95%"], "p99_ms": agg["99%"],
        "app_cpu_avg_%": round(cpu_summary[users][0], 1),
        "app_cpu_max_%": round(cpu_summary[users][1], 1),
    })
    e = df[df["Name"] != "Aggregated"][["Name","Request Count","Failure Count","Requests/s","50%","95%","99%"]].copy()
    e.insert(0, "users", users); per_endpoint.append(e)

summary = pd.DataFrame(rows)
endpoints = pd.concat(per_endpoint)
summary.to_csv(f"{RESULTS}/summary.csv", index=False)
endpoints.to_csv(f"{RESULTS}/per_endpoint.csv", index=False)
display(summary); display(endpoints)
print(summary.to_markdown(index=False))   # dán thẳng vào README

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(summary["users"], summary["RPS"], marker="o"); ax[0].set(title="Throughput", xlabel="Concurrent users", ylabel="req/s")
for p in ["p50_ms", "p95_ms", "p99_ms"]:
    ax[1].plot(summary["users"], summary[p], marker="o", label=p)
ax[1].set(title="Latency", xlabel="Concurrent users", ylabel="ms"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/loadtest_charts.png", dpi=150); plt.show()

## 10b. Burst test – 100 / 200 / 500 / 1000 request cùng lúc

Với mỗi mức N, tạo N user cùng lúc (`-r N`), mỗi user gửi đúng 1 request rồi dừng; chạy lần lượt cho 4 target `menu`, `register`, `login`, `order`.
Phần tạo tài khoản/token chạy trong `prepare()` trước khi đo, không tính vào kết quả (mức 1000 có thể mất vài phút vì BCrypt).
Script: `loadtest/locustfile_burst.py` (lấy từ repo, nhánh `dev`).


In [ ]:
BURST_FILE = f"{APP_DIR}/loadtest/locustfile_burst.py"
BURST_LEVELS  = [100, 200, 500, 1000]          # số request mỗi lần burst
BURST_TARGETS = ["menu", "register", "login", "order"]

import pandas as pd
rows = []
for n in BURST_LEVELS:
    for target in BURST_TARGETS:
        tag = f"burst_{target}_{n}"
        print(f"=== burst: {target} x {n} ===")
        subprocess.run(
            f"BURST_TARGET={target} BURST_REQUESTS={n} RUN_ID=b{target}{n} "
            f"taskset -c {LOCUST_CORES} locust -f {BURST_FILE} --host {BASE} --headless "
            f"-u {n} -r {n} --csv {RESULTS}/{tag} --only-summary",
            shell=True)
        df = pd.read_csv(f"{RESULTS}/{tag}_stats.csv")
        a = df[df["Name"] == "Aggregated"].iloc[0]
        rows.append({"requests": n, "target": target,
                     "done": int(a["Request Count"]), "failures": int(a["Failure Count"]),
                     "avg_ms": round(a["Average Response Time"], 1),
                     "p50_ms": a["50%"], "p95_ms": a["95%"], "p99_ms": a["99%"],
                     "max_ms": round(a["Max Response Time"], 1)})
        time.sleep(10)   # nghỉ giữa các lần burst

burst_summary = pd.DataFrame(rows)
burst_summary.to_csv(f"{RESULTS}/burst_summary.csv", index=False)
display(burst_summary)
print(burst_summary.to_markdown(index=False))


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for t in BURST_TARGETS:
    d = burst_summary[burst_summary["target"] == t]
    ax[0].plot(d["requests"], d["p95_ms"], marker="o", label=t)
    ax[1].plot(d["requests"], d["failures"], marker="o", label=t)
ax[0].set(title="Burst p95 latency", xlabel="Requests cùng lúc", ylabel="ms", xscale="log"); ax[0].legend()
ax[1].set(title="Burst failures", xlabel="Requests cùng lúc", ylabel="số lỗi", xscale="log"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{RESULTS}/burst_charts.png", dpi=150); plt.show()


## 11. Dọn dẹp

In [ ]:
app.terminate(); subprocess.run("mysqladmin -uroot shutdown", shell=True)
!ls -lh {RESULTS}